# 🛡️ PEDAS 2026 — Feature Engineering Pipeline (v2)
**Platform Evaluasi Data Sains (PEDAS) 2026**  
*Pipeline: Data Loading ➔ Label Cleaning ➔ IP Validation ➔ Mislabel Audit ➔ Confidence Fix ➔ Datetime Features ➔ IP Features ➔ URL Features ➔ Brand Features ➔ Domain Frequency ➔ Baseline Modeling*
---


## 1. Setup Lingkungan & Import Library
Mengimpor dependensi yang dibutuhkan serta modul `audit_mislabel.py` dari parent directory.


In [6]:
import sys
import os
import re
import pandas as pd
sys.path.insert(0, '../')

from audit_mislabel import audit_mislabel, apply_relabel, summarize_audit, summarize_diff, recompute_confidence


## 2. Pemuatan Dataset (Train & Test)
Memuat file data mentah `training.csv` dan `predict.csv` dari folder `../../data/raw/`.


In [7]:
# ============================================================
# CELL 2: Ambil data
# ============================================================
train = pd.read_csv('../../data/raw/training.csv')
test  = pd.read_csv('../../data/raw/predict.csv')

print(f"Dimensi Training : {train.shape}")
print(f"Dimensi Predict  : {test.shape}")
train.head(3)

Dimensi Training : (8400, 10)
Dimensi Predict  : (1500, 10)


,url,brand,discovered,confidence_level,ip,domain,sld,category,registrar,registration_date
0,https://lucah3.***********.my.id/,Telegram,5/1/2024 21:52,100,NaN,***********.my.id,my.id,phishingg,PT Web Media Technology Indonesia,5/14/2026
1,https://dpmptsp.*********.go.id/petapotensi/da...,judi online,8/5/2024 20:41,100,103.162.68.84,*********.go.id,go.id,online gambling,Kementerian Komunikasi dan Informatika,5/11/2009
2,http://klikpad.bkpd.**************.go.id/klikp...,-,6/14/2024 7:05,100,103.18.117.8,**************.go.id,go.id,online gambling,Kementerian Komunikasi dan Informatika,3/13/2008


## 3. Pembersihan Variasi Penulisan Kategori
Standarisasi penulisan nilai target `category` pada data train (lowercase, strip whitespace, dan pemetaan typo).


In [8]:
cat_map = {
    'online gamblingg': 'online gambling', 'Online Gambling': 'online gambling',
    'phishingg': 'phishing', 'otherr': 'other', 'Other': 'other',
    'malwaree': 'malware', 'spamm': 'spam', 'Brand': 'brand',
    'FakeShop': 'fakeshop', 'PIIExposure': 'pii_exposure',
}
train['category'] = train['category'].str.strip().replace(cat_map).str.lower().str.strip()

print("=== Distribusi Kategori Setelah Pembersihan ===")
print(train['category'].value_counts())

=== Distribusi Kategori Setelah Pembersihan ===
category
online gambling    5447
phishing           2253
other               284
spam                185
malware             179
brand                45
fakeshop              5
violence              1
pii_exposure          1
Name: count, dtype: int64


## 4. Audit Potensi Mislabeling
Menjalankan `audit_mislabel()` untuk menghitung skor kecurigaan anomali label berdasarkan bukti URL dan IP.


In [9]:
# ============================================================
# CELL 4: Audit & perbaiki category (pakai bukti url + ip)
# ============================================================
train_audited = audit_mislabel(train)
summarize_audit(train_audited)

=== Distribusi skor kecurigaan ===
suspect_score
11       1
8       24
7        1
6        1
5       34
4       10
3      129
2      134
1      132
0     7934
Name: count, dtype: int64

Skor >=5 (relabel otomatis)      : 61 baris
Skor 3-4 (verifikasi manual)     : 139 baris
Skor 1-2 (biarkan, terlalu lemah): 266 baris


## 5. Perbaikan Anomali `confidence_level`
Merekalkulasi nilai kepercayaan menggunakan bukti yang ditemukan saat proses audit.


In [10]:
# ============================================================
# CELL 5: Perbaiki confidence_level yang anomali (reuse bukti dari audit)
# ============================================================
train_audited = recompute_confidence(train_audited)

print("Distribusi confidence_level setelah diperbaiki:")
print(train_audited['confidence_level'].value_counts().sort_index())

Distribusi confidence_level setelah diperbaiki:
confidence_level
0        16
40       18
50       23
60        3
90      209
100    8131
Name: count, dtype: int64


## 6. Penerapan Relabeling Otomatis
Menerapkan relabel kategori untuk baris dengan `suspect_score >= 5` dan mengekspor perubahannya ke `relabel_diff.csv`.


In [11]:
# ============================================================
# CELL 6: Terapkan relabel category (skor >= 5)
# ============================================================
train, diff_table = apply_relabel(train_audited, score_threshold=5)
summarize_diff(diff_table)

diff_table.to_csv('../../data/result/v2/relabel_diff.csv', index=False)

=== Total baris yang benar-benar berubah label: 61 ===

=== Perubahan per pasangan (label lama -> label baru) ===
category_before  category_after 
phishing         online gambling    20
malware          online gambling    14
spam             online gambling    12
other            online gambling     8
online gambling  phishing            6
spam             phishing            1
dtype: int64


## 7. Normalisasi Skala `confidence_level`
Menskalakan nilai confidence ke rentang 0 hingga 1 (`confidence_norm`).


In [12]:
# ============================================================
# CELL 7: Normalisasi confidence_level ke 0-1
# ============================================================
train['confidence_norm'] = train['confidence_level'] / 100

## 8. Ekstraksi Fitur Tanggal & Waktu
Konversi kolom waktu ke objek datetime dan ekstraksi jam serta hari dalam seminggu.


In [13]:
# ============================================================
# CELL 8: Ubah format tanggal
# ============================================================
train['discovered'] = pd.to_datetime(train['discovered'], format='%m/%d/%Y %H:%M')
train['registration_date'] = pd.to_datetime(train['registration_date'], format='%m/%d/%Y')
test['discovered'] = pd.to_datetime(test['discovered'], format='%m/%d/%Y %H:%M')
test['registration_date'] = pd.to_datetime(test['registration_date'], format='%m/%d/%Y')

train['discovered_hour'] = train['discovered'].dt.hour
train['discovered_dayofweek'] = train['discovered'].dt.dayofweek
test['discovered_hour'] = test['discovered'].dt.hour
test['discovered_dayofweek'] = test['discovered'].dt.dayofweek

## 9. Rekayasa Fitur Umur Domain
Menghitung selisih hari (`domain_age_days`) dan membuat penanda anomali tanggal negatif (`domain_age_is_negative`).


In [14]:
# ============================================================
# CELL 9: Selisih discovered vs registration_date
# ============================================================
train['domain_age_days'] = (train['discovered'] - train['registration_date']).dt.days
test['domain_age_days'] = (test['discovered'] - test['registration_date']).dt.days

# JANGAN di-clip! Nilai negatif adalah sinyal kuat ke phishing (sudah terbukti di EDA)
train['domain_age_is_negative'] = (train['domain_age_days'] < 0).astype(int)
test['domain_age_is_negative'] = (test['domain_age_days'] < 0).astype(int)

## 10. Fitur Jaringan & Infrastruktur IP
Menghitung frekuensi kemunculan IP dan subnet `/24` berbasis referensi data train untuk mencegah data leakage.


In [15]:
# ============================================================
# CELL 10: Uraikan IP (ip_frequency, ip_subnet_frequency, ip_is_missing)
# WAJIB: hitung referensi HANYA dari train, terapkan ke train & test
# ============================================================
ip_freq_map = train['ip'].value_counts().to_dict()

train['ip_prefix24'] = train['ip'].str.rsplit('.', n=1).str[0]
test['ip_prefix24'] = test['ip'].str.rsplit('.', n=1).str[0]
subnet_freq_map = train['ip_prefix24'].value_counts().to_dict()

train['ip_frequency'] = train['ip'].map(ip_freq_map).fillna(0)
train['ip_subnet_frequency'] = train['ip_prefix24'].map(subnet_freq_map).fillna(0)
train['ip_is_missing'] = train['ip'].isna().astype(int)

test['ip_frequency'] = test['ip'].map(ip_freq_map).fillna(0)
test['ip_subnet_frequency'] = test['ip_prefix24'].map(subnet_freq_map).fillna(0)
test['ip_is_missing'] = test['ip'].isna().astype(int)

train = train.drop(columns=['ip_prefix24'])
test = test.drop(columns=['ip_prefix24'])

## 11. Ekstraksi Fitur Karakteristik URL
Mengekstrak panjang URL dan mendeteksi keberadaan pola encoding Base64.


In [16]:
# ============================================================
# CELL 11: Fitur dari URL
# ============================================================
train['url_length'] = train['url'].str.len()
test['url_length'] = test['url'].str.len()

train['url_has_base64'] = train['url'].str.contains(r'[A-Za-z0-9+/_-]{40,}={0,2}', regex=True)
test['url_has_base64'] = test['url'].str.contains(r'[A-Za-z0-9+/_-]{40,}={0,2}', regex=True)

## 12. Penanganan Fitur Brand
Menandai missing value pada kolom brand dan mengisi nilai default `unknown`.


In [17]:
# ============================================================
# CELL 12: Brand -- flag missing (termasuk '-'), isi 'unknown'
# ============================================================
train['brand_is_missing'] = train['brand'].isna() | (train['brand'].str.strip() == '-')
test['brand_is_missing'] = test['brand'].isna() | (test['brand'].str.strip() == '-')

for df in [train, test]:
    df['brand'] = df['brand'].fillna('unknown')
    df.loc[df['brand'].str.strip() == '-', 'brand'] = 'unknown'
    df['brand'] = df['brand'].str.lower().str.strip()

## 13. Frekuensi Domain & One-Hot Encoding SLD
Menghitung frekuensi domain serta melakukan One-Hot Encoding pada fitur SLD.


In [18]:
# ============================================================
# CELL 13: Domain frequency + registrar + sld
# ============================================================
domain_freq_map = train['domain'].value_counts().to_dict()
train['domain_frequency'] = train['domain'].map(domain_freq_map).fillna(0)
test['domain_frequency'] = test['domain'].map(domain_freq_map).fillna(0)

train['registrar'] = train['registrar'].fillna('unknown').str.strip()
test['registrar'] = test['registrar'].fillna('unknown').str.strip()

# sld: one-hot encoding
train = pd.get_dummies(train, columns=['sld'], prefix='sld')
test = pd.get_dummies(test, columns=['sld'], prefix='sld')

## 14. Pengecekan Kualitas Data (Sanity Check)
Memastikan tidak ada nilai null yang tersisa pada kolom fitur yang akan digunakan oleh model.


In [19]:
# ============================================================
# CELL 14: Cek sanity -- pastikan tidak ada missing value tersisa
# di kolom-kolom final yang akan dipakai model
# ============================================================
feature_cols = ['confidence_norm', 'discovered_hour', 'discovered_dayofweek',
                'domain_age_days', 'domain_age_is_negative',
                'ip_frequency', 'ip_subnet_frequency', 'ip_is_missing',
                'url_length', 'url_has_base64', 'brand_is_missing', 'domain_frequency']

print("Missing value tersisa (train):")
print(train[feature_cols].isna().sum()[train[feature_cols].isna().sum() > 0])

Missing value tersisa (train):
domain_age_days    21
dtype: int64


## 15. Ekspor Hasil Preprocessing
Menyimpan dataframe akhir ke folder `../../data/result/v2/`.


In [20]:
# ============================================================
# CELL 15: Simpan hasil cleaning
# ============================================================
os.makedirs('../../data/result/v2', exist_ok=True)
train.to_csv('../../data/result/v2/train_final.csv', index=False)
test.to_csv('../../data/result/v2/test_final.csv', index=False)
print("Tersimpan!")

Tersimpan!


## 16. Evaluasi Baseline Model
Validasi performa model menggunakan Stratified Train-Validation Split dan Random Forest Classifier.


In [ ]:
# ============================================================
# CELL 16: Split train/validation & training baseline model
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

sld_cols = [c for c in train.columns if c.startswith('sld_')]
X = train[feature_cols + sld_cols]
y = train['category']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

print(classification_report(y_val, model.predict(X_val), zero_division=0))

                 precision    recall  f1-score   support

          brand       1.00      0.18      0.31        11
       fakeshop       0.00      0.00      0.00         1
        malware       0.75      0.54      0.62        28
online gambling       0.97      0.99      0.98      1076
          other       1.00      0.93      0.96        55
       phishing       0.99      0.96      0.97       483
           spam       0.79      0.88      0.84        26

       accuracy                           0.97      1680
      macro avg       0.78      0.64      0.67      1680
   weighted avg       0.97      0.97      0.96      1680



c:\syntax\Codelab\Aptikom-Pedas2026\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\syntax\Codelab\Aptikom-Pedas2026\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\syntax\Codelab\Aptikom-Pedas2026\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is